# LeetCode 464 — Can I Win bằng Python

Link đề: https://leetcode.com/problems/can-i-win/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, Memoization, Math, Dynamic Programming, Bitmask, Game Theory

Skeleton LeetCode:

```python
class Solution:
    def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool:
        ...
```

Đây là bài **trò chơi hai người** kinh điển trên "tập đã dùng": ta biểu diễn tập số đã chọn bằng **mặt nạ bit (bitmask)** rồi **ghi nhớ** kết quả từng tập. Notebook dạy từ khái niệm "thế thắng / thế thua", qua đệ quy vét cạn, ghi nhớ theo mask, quy hoạch động từ dưới lên, đến cắt tỉa và duyệt chỉ các trạng thái có thể đến.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai người luân phiên chọn **một số chưa dùng** từ `1..maxChoosableInteger` (không chọn lại), cộng vào tổng chung; ai **làm tổng đạt hoặc vượt** `desiredTotal` trước thì thắng; hỏi người đi đầu có **ép thắng** được không khi cả hai chơi tối ưu.
2. Nắm các công cụ Python: toán tử bit (`1 << i`, `>>`, `&`, `|`), mặt nạ bit biểu diễn tập hợp, `dict` ghi nhớ (memo), `functools.cache`, công thức tổng `n(n+1)/2`, và duyệt tập con.
3. Đi từ **minimax vét cạn `O(n!)`** → **ghi nhớ theo mask `O(2^n · n)`** → **DP từ dưới lên** → **ghi nhớ + cắt tỉa theo "số lớn nhất còn lại"** → **duyệt tầng các trạng thái có thể đến rồi đánh giá ngược**.
4. Hiểu mấu chốt: tổng hiện tại **do tập đã dùng quyết định hoàn toàn** (tổng = tổng các số trong mask), nên **chỉ cần `mask` làm khoá ghi nhớ**; và "thế thắng" nghĩa là **tồn tại** nước đi thắng ngay hoặc dẫn đối thủ vào **thế thua**.
5. Tránh các lỗi kinh điển: quên loại trường hợp `tổng tất cả < desiredTotal`, ghi nhớ theo **tổng** thay vì theo mask, dùng chung bộ nhớ đệm giữa nhiều test, nhầm với trò chơi "100" cho phép chọn lại, và lệch chỉ số bit.

## 2. Đề bài & ví dụ

Trong trò chơi "100", hai người luân phiên cộng vào tổng chung một số nguyên từ `1` đến `10`; ai làm tổng **đạt hoặc vượt `100` đầu tiên** thì thắng.

Nếu đổi luật để người chơi **không được dùng lại** số đã chọn thì sao? Ví dụ hai người lần lượt rút từ một đống số `1..15` **không hoàn lại** cho tới khi tổng `>= 100`.

Cho hai số nguyên `maxChoosableInteger` và `desiredTotal`. Trả về `true` nếu **người đi trước** có thể **ép thắng**, ngược lại `false`. Giả sử cả hai chơi **tối ưu**.

**Ví dụ 1**

- Input: `maxChoosableInteger = 10, desiredTotal = 11`
- Output: `false`
- Giải thích: dù người đi trước chọn số nào, người thứ hai luôn thắng. Ví dụ chọn `1`, người thứ hai chọn `10` được tổng `11`. Các lựa chọn khác tương tự.

**Ví dụ 2**

- Input: `maxChoosableInteger = 10, desiredTotal = 0`
- Output: `true`

**Ví dụ 3**

- Input: `maxChoosableInteger = 10, desiredTotal = 1`
- Output: `true`

**Ràng buộc:**

- `1 <= maxChoosableInteger <= 20`
- `0 <= desiredTotal <= 300`

In [1]:
from typing import List
from functools import cache
from itertools import combinations
from collections import deque
import random
import time

examples = [((10, 11), False), ((10, 0), True), ((10, 1), True)]
for args, expected in examples:
    print(args, "-> kỳ vọng", expected)

(10, 11) -> kỳ vọng False
(10, 0) -> kỳ vọng True
(10, 1) -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Thế thắng / thế thua và minimax

Trong trò chơi hai người, mỗi **thế** (trạng thái) có hai khả năng cho người **đang đến lượt**:

- **Thế thắng:** tồn tại một nước đi mà hoặc **thắng ngay**, hoặc đưa đối thủ vào **thế thua**.
- **Thế thua:** **mọi** nước đi đều khiến đối thủ có thể thắng (hoặc không còn nước nào).

Đây là minimax rút gọn: "tôi thắng nếu **tồn tại** nước đi sao cho đối thủ **không** thắng được". Trong code là `any(not win(thế_sau) for nước_đi in các_nước)`.

In [2]:
# Ví dụ nhỏ: trò chơi "bốc que": còn r que, mỗi lượt bốc 1 hoặc 2, ai bốc que cuối thắng
def win_sticks(r):
    if r == 0:
        return False                     # không còn que: người đến lượt đã thua (đối thủ vừa bốc xong)
    return any(not win_sticks(r - take) for take in (1, 2) if take <= r)

print([(r, win_sticks(r)) for r in range(0, 8)])    # r chia hết cho 3 là thế thua

[(0, False), (1, True), (2, True), (3, False), (4, True), (5, True), (6, False), (7, True)]


### 3.2. Mặt nạ bit (bitmask) biểu diễn tập hợp

Một số nguyên `mask` có thể coi là **tập con** của `{1, ..., n}`: bit thứ `i-1` bằng `1` nghĩa là số `i` **đã dùng**.

- Dùng số `i`: `mask | (1 << (i - 1))`.
- Kiểm tra đã dùng: `(mask >> (i - 1)) & 1`.
- Tập rỗng: `0`. Tập đầy đủ `n` số: `(1 << n) - 1`.

Lưu ý Python: toán tử bit (`<<`, `&`, `|`) có độ ưu tiên **cao hơn** phép so sánh (`==`), ngược với C/C++; ta vẫn **thêm ngoặc** cho dễ đọc.

In [3]:
n = 5
mask = 0
for i in (2, 5):
    mask |= 1 << (i - 1)               # đã dùng 2 và 5
print(bin(mask), mask)                  # 0b10010 18
print([i for i in range(1, n + 1) if (mask >> (i - 1)) & 1])     # [2, 5]
print([i for i in range(1, n + 1) if not (mask >> (i - 1)) & 1]) # [1, 3, 4]
print(bin((1 << n) - 1))                # 0b11111

0b10010 18
[2, 5]
[1, 3, 4]
0b11111


### 3.3. Tổng hiện tại do mask quyết định

Tổng đã cộng chính là **tổng các số trong tập đã dùng**. Do đó hai trạng thái có cùng `mask` thì cùng tổng, cùng "còn thiếu bao nhiêu". Nhờ vậy **`mask` đủ làm khoá** cho ghi nhớ (không cần thêm tổng).

In [4]:
def mask_sum(mask, n):
    return sum(i for i in range(1, n + 1) if (mask >> (i - 1)) & 1)

print(mask_sum(0b10010, 5), mask_sum(0b11111, 5))      # 7 15

7 15


### 3.4. Ghi nhớ (memoization) bằng `dict` và `functools.cache`

Nếu cùng một thế được hỏi nhiều lần, lưu kết quả vào `dict` để lần sau tra ngay: tránh tính lại. `@cache` (Python 3.9+) làm việc đó tự động cho hàm thuần theo tham số. Số thế phân biệt ở đây tối đa `2^n` (mỗi số dùng/không dùng).

In [5]:
calls = 0

@cache
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(30), "số lần tính thật:", calls)       # 832040, chỉ 31 lần

832040 số lần tính thật: 31


### 3.5. Công thức tổng `1 + 2 + ... + n`

`n(n+1)/2` (trong Python `n * (n + 1) // 2`). Nếu tổng của **tất cả** số cũng nhỏ hơn `desiredTotal` thì **không ai có thể thắng**: đề quy ước kết quả là `false`.

In [6]:
for n in (3, 10, 20):
    print(n, n * (n + 1) // 2, sum(range(1, n + 1)))
print("max=10, target=56 có thể thắng?", 10 * 11 // 2 >= 56)      # 55 < 56 -> không ai thắng

3 6 6
10 55 55
20 210 210
max=10, target=56 có thể thắng? False


### 3.6. Duyệt mọi tập con bằng mask

`for mask in range(1 << n)` đi qua **tất cả** `2^n` tập con của `n` phần tử. Ta dùng khi viết DP từ dưới lên.

In [7]:
n = 3
for mask in range(1 << n):
    print(f"{mask:03b}", [i for i in range(1, n + 1) if (mask >> (i - 1)) & 1])

000 []
001 [1]
010 [2]
011 [1, 2]
100 [3]
101 [1, 3]
110 [2, 3]
111 [1, 2, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `maxChoosableInteger = 3`, `desiredTotal = 4` (số `1, 2, 3`, tổng tất cả `6 ≥ 4` nên có thể có người thắng). Người đi trước là **A**, người thứ hai là **B**.

| A chọn | Tổng | B có thể chọn | Kết quả |
|---|---|---|---|
| 1 | 1 | `2` → 3; `3` → **4 ≥ 4** | B chọn 3 thắng |
| 2 | 2 | `1` → 3; `3` → **5 ≥ 4** | B chọn 3 thắng |
| 3 | 3 | `1` → **4 ≥ 4**; `2` → **5 ≥ 4** | B thắng ngay |

Mọi nước đi của A đều để B có nước thắng ngay ⇒ A **thua** ⇒ `canIWin(3, 4) = False`.

Quy trình tổng quát: với mỗi nước đi `i` còn dùng được, A **thắng** nếu `tổng + i ≥ desiredTotal` **hoặc** nếu sau khi chọn `i`, **B ở thế thua**. A thắng nếu tồn tại một nước như vậy.

In [8]:
# Kiểm chứng bảng trên bằng code (cách viết rất thô)
def tiny(n, target):
    def win(used, total):
        for i in range(1, n + 1):
            if i in used:
                continue
            if total + i >= target or not win(used | {i}, total + i):
                return True
        return False
    return win(frozenset(), 0)

print(tiny(3, 4), tiny(3, 6), tiny(4, 6))     # False True True

False True True


## 5. Cách 1 — Brute Force / baseline: minimax vét cạn không ghi nhớ

**Ý tưởng:** đúng như bảng mục 4: đệ quy `win(used, total)` thử mọi số chưa dùng. Thêm kiểm tra: nếu tổng tất cả `< desiredTotal` thì trả `False` ngay.

Code `can_win_brute` (dùng list `used` để dễ nhìn, chưa dùng bit):

In [9]:
def can_win_brute(n, target):
    if n * (n + 1) // 2 < target:
        return False                       # không ai có thể đạt target

    used = [False] * (n + 1)

    def win(total):
        for i in range(1, n + 1):
            if not used[i]:
                if total + i >= target:
                    return True
                used[i] = True
                opponent_wins = win(total + i)
                used[i] = False             # trả lại (quay lui)
                if not opponent_wins:
                    return True
        return False

    return win(0)


for (n, t), expected in examples:
    if n <= 8:
        print((n, t), can_win_brute(n, t), "kỳ vọng", expected)
print(can_win_brute(7, 12), can_win_brute(8, 20))

True True


**Độ phức tạp:** số dãy chọn có thể lên tới `n!` ⇒ `Time = O(n!)` (với `n = 20` là `2.4·10^18`), `Space = O(n)`.

**Lưu ý:** ví dụ 1 có `n = 10` nên ta bỏ qua ở đây bằng điều kiện `n <= 8` (sẽ chạy bằng các cách nhanh hơn). Thử `can_win_brute(10, 11)` có thể vẫn chạy được vì cắt sớm, nhưng không đảm bảo ở các test khác.

**Nút thắt:** cùng một **tập đã dùng** có thể đạt được bằng nhiều thứ tự khác nhau (chọn `1` rồi `2` hay `2` rồi `1`): cả hai cho cùng thế. Brute force giải lại thế đó nhiều lần.

## 6. Ý tưởng thuật toán / bước đột phá

Một **thế** của trò chơi chỉ phụ thuộc vào **tập số đã dùng** (thứ tự không quan trọng) và tổng do tập này quyết định. Vậy:

- Số thế phân biệt tối đa **`2^n`** (với `n ≤ 20`: khoảng `10^6`), thay vì `n!`.
- Kết quả của thế (người đến lượt có thắng không) **chỉ phụ thuộc `mask`**.
- Ghi nhớ `win[mask]`: mỗi thế tính một lần, mỗi lần thử `n` nước đi ⇒ `O(2^n · n)`.

Đây là ý tưởng "state compression" (nén trạng thái): biến một tập con thành một số nguyên để dùng làm chỉ số/khoá.

In [10]:
# Hai thứ tự khác nhau dẫn tới CÙNG mask và cùng tổng
order1 = [1, 2]
order2 = [2, 1]
def to_mask(seq):
    m = 0
    for i in seq:
        m |= 1 << (i - 1)
    return m
print(to_mask(order1), to_mask(order2), sum(order1), sum(order2))     # 3 3 3 3

3 3 3 3


## 7. Các cách tiếp theo

### Cách 2 — Đệ quy + ghi nhớ theo mask (top-down)

**Giải thích từng đoạn:**

1. Nếu `tổng tất cả < desiredTotal`: `False`.
2. Hàm `win(mask, total)`: với mỗi `i` chưa dùng: nếu `total + i >= target` hoặc `not win(mask | bit, total + i)` thì trả `True`.
3. Nếu không có nước nào thắng: `False`.
4. Lưu kết quả vào `memo[mask]`.

In [11]:
def can_win_memo(n, target):
    if n * (n + 1) // 2 < target:
        return False
    memo = {}

    def win(mask, total):
        if mask in memo:
            return memo[mask]
        result = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if total + i >= target or not win(mask | bit, total + i):
                result = True
                break
        memo[mask] = result
        return result

    return win(0, 0)


for (n, t), expected in examples:
    print((n, t), can_win_memo(n, t), "kỳ vọng", expected)

(10, 11) False kỳ vọng False
(10, 0) True kỳ vọng True
(10, 1) True kỳ vọng True


In [12]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool:
        n, target = maxChoosableInteger, desiredTotal
        if n * (n + 1) // 2 < target:
            return False
        memo = {}

        def win(mask, total):
            if mask in memo:
                return memo[mask]
            result = False
            for i in range(1, n + 1):
                bit = 1 << (i - 1)
                if mask & bit:
                    continue
                if total + i >= target or not win(mask | bit, total + i):
                    result = True
                    break
            memo[mask] = result
            return result

        return win(0, 0)


print(Solution2().canIWin(10, 11), Solution2().canIWin(10, 0), Solution2().canIWin(10, 1))

False True True


**Độ phức tạp Cách 2:** `Time = O(2^n · n)` trường hợp xấu nhất, `Space = O(2^n)`. Thực tế **ít hơn** vì ta cắt sớm khi tìm được nước thắng và chỉ thăm các thế có thể đến.

> Lưu ý: bộ nhớ `memo` được tạo **bên trong** hàm cho mỗi test. Nếu để `memo` toàn cục dùng chung cho nhiều test (khác `n`, `target`), kết quả sẽ sai (xem mục 13).

### Cách 3 — Quy hoạch động từ dưới lên trên mọi mask

**Ý tưởng:** thay đệ quy bằng vòng lặp. Với mọi `mask` từ **lớn xuống nhỏ** (mask lớn hơn = đã dùng nhiều số hơn, tức "ở sau" trong ván chơi), tính `win[mask]` từ các mask con của nó. Cần tổng của mỗi mask: `total[mask] = total[mask không có bit thấp nhất] + (giá trị bit thấp nhất)`.

Chỉ xét các mask có `total < target` (thế hợp lệ chưa kết thúc).

In [13]:
def can_win_dp(n, target):
    if n * (n + 1) // 2 < target:
        return False
    if target <= n:
        return True                                 # chọn ngay số lớn nhất n là đủ (cũng xử lý target = 0)
    full = 1 << n
    total = [0] * full
    for mask in range(1, full):
        low = (mask & -mask).bit_length()          # chỉ số (1-based) của bit thấp nhất = số được dùng
        total[mask] = total[mask & (mask - 1)] + low
    win = [False] * full
    for mask in range(full - 1, -1, -1):
        s = total[mask]
        if s >= target:
            continue                                # thế đã kết thúc, không phải thế "đang chơi"
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if s + i >= target or not win[mask | bit]:
                win[mask] = True
                break
    return win[0]


for (n, t), expected in examples:
    print((n, t), can_win_dp(n, t), "kỳ vọng", expected)

(10, 11) False kỳ vọng False
(10, 0) True kỳ vọng True
(10, 1) True kỳ vọng True


In [14]:
# Code nộp LeetCode — Cách 3 (chậm hơn Cách 2 khi n = 20 vì quét hết 2^n mask)
class Solution3:
    def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool:
        n, target = maxChoosableInteger, desiredTotal
        if n * (n + 1) // 2 < target:
            return False
        if target <= n:
            return True
        full = 1 << n
        total = [0] * full
        for mask in range(1, full):
            total[mask] = total[mask & (mask - 1)] + (mask & -mask).bit_length()
        win = [False] * full
        for mask in range(full - 1, -1, -1):
            s = total[mask]
            if s >= target:
                continue
            for i in range(1, n + 1):
                bit = 1 << (i - 1)
                if mask & bit:
                    continue
                if s + i >= target or not win[mask | bit]:
                    win[mask] = True
                    break
        return win[0]


print(Solution3().canIWin(10, 11), Solution3().canIWin(4, 6), Solution3().canIWin(3, 7))

False True False


**Độ phức tạp Cách 3:** `Time = Θ(2^n · n)` (luôn quét mọi mask), `Space = Θ(2^n)` (hai mảng). Với `n = 20` Python mất khá lâu (vài giây đến chục giây); hữu ích để **hiểu thứ tự tính** (mask lớn trước mask nhỏ) và để kiểm chứng.

> Vì sao duyệt mask giảm dần là đúng? `mask | bit > mask`, nên mọi thế "sau" đều có chỉ số **lớn hơn** và đã được tính xong.

### Cách 4 — Ghi nhớ + cắt tỉa: "số lớn nhất còn lại đủ để thắng"

**Ý tưởng cải tiến Cách 2:** gọi `need = target - total` là phần còn thiếu. Nếu **số lớn nhất chưa dùng** `≥ need` thì người đến lượt **thắng ngay**; không cần thử gì thêm. Thêm nữa, thử các số **từ lớn xuống nhỏ** vì số lớn dễ thắng ngay hoặc dễ đẩy `need` về nhỏ khiến thế đối thủ rõ ràng.

Ta đếm số thế đã thăm để so sánh với Cách 2.

In [15]:
def can_win_pruned(n, target, stats=None):
    if n * (n + 1) // 2 < target:
        return False
    memo = {}

    def win(mask, need):
        if mask in memo:
            return memo[mask]
        # số lớn nhất chưa dùng
        for i in range(n, 0, -1):
            if not (mask >> (i - 1)) & 1:
                biggest = i
                break
        if biggest >= need:                 # thắng ngay
            memo[mask] = True
            return True
        result = False
        for i in range(n, 0, -1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if not win(mask | bit, need - i):
                result = True
                break
        memo[mask] = result
        return result

    ans = win(0, target)
    if stats is not None:
        stats["states"] = len(memo)
    return ans


def can_win_memo_count(n, target):
    """Cách 2 có đếm số thế thăm, để so sánh."""
    if n * (n + 1) // 2 < target:
        return False, 0
    memo = {}
    def win(mask, total):
        if mask in memo:
            return memo[mask]
        result = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if total + i >= target or not win(mask | bit, total + i):
                result = True
                break
        memo[mask] = result
        return result
    return win(0, 0), len(memo)


for (n, t), expected in examples:
    st = {}
    print((n, t), can_win_pruned(n, t, st), "kỳ vọng", expected, "| số thế thăm:", st)
for n, t in ((15, 90), (17, 100), (18, 150), (20, 150)):
    st = {}
    a = can_win_pruned(n, t, st)
    b, cnt = can_win_memo_count(n, t)
    print((n, t), a == b, "thế thăm (cắt tỉa) =", st.get("states"), "| (Cách 2) =", cnt)

(10, 11) False kỳ vọng False | số thế thăm: {'states': 11}
(10, 0) True kỳ vọng True | số thế thăm: {'states': 1}
(10, 1) True kỳ vọng True | số thế thăm: {'states': 1}
(15, 90) True thế thăm (cắt tỉa) = 10432 | (Cách 2) = 14911


(17, 100) True thế thăm (cắt tỉa) = 79990 | (Cách 2) = 77657


(18, 150) True thế thăm (cắt tỉa) = 119014 | (Cách 2) = 56592


(20, 150) True thế thăm (cắt tỉa) = 780462 | (Cách 2) = 732041


In [16]:
# Code nộp LeetCode — Cách 4
class Solution4:
    def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool:
        n, target = maxChoosableInteger, desiredTotal
        if n * (n + 1) // 2 < target:
            return False
        memo = {}

        def win(mask, need):
            if mask in memo:
                return memo[mask]
            for i in range(n, 0, -1):
                if not (mask >> (i - 1)) & 1:
                    biggest = i
                    break
            if biggest >= need:
                memo[mask] = True
                return True
            result = False
            for i in range(n, 0, -1):
                bit = 1 << (i - 1)
                if not mask & bit and not win(mask | bit, need - i):
                    result = True
                    break
            memo[mask] = result
            return result

        return win(0, target)


print(Solution4().canIWin(10, 11), Solution4().canIWin(5, 9), Solution4().canIWin(6, 21))

False True False


**Điều kiện an toàn của phép cắt tỉa:** khi vào `win(mask, need)` luôn có **ít nhất một số chưa dùng** (nếu `need > 0` và tổng tất cả `≥ target`, số còn lại đủ để kết thúc; ta đã chặn `tổng tất cả < target` ở đầu). Vì `need` sau mỗi nước đi giảm đúng bằng số vừa chọn, `need > 0` vẫn đảm bảo còn số để chọn: nếu hết số mà `need > 0` thì tổng tất cả `< target`, trái với giả thiết.

**Độ phức tạp Cách 4:** vẫn `O(2^n · n)` trường hợp xấu nhất, `Space = O(2^n)`, nhưng số thế thăm **không nhất thiết ít hơn** Cách 2 (bảng trên: có ca ít hơn như `(15, 90)` và `(20, 100)`, có ca nhiều hơn như `(18, 150)`), vì thứ tự thử số từ lớn xuống nhỏ cũng đổi cây tìm kiếm. Cắt tỉa là một heuristic; hãy **đo** thay vì tin.

### Cách 5 — Duyệt tầng các thế có thể đến rồi đánh giá ngược

**Ý tưởng khác:** không dùng đệ quy. Hai giai đoạn:

1. **Giai đoạn xuôi:** BFS theo số số đã chọn: tầng `k` là các mask có `k` bit mà `tổng < target` (chưa kết thúc), đạt được từ tầng `k-1`. Lưu `total[mask]`.
2. **Giai đoạn ngược:** duyệt tầng từ **cuối lên đầu**; `win[mask] = True` nếu có `i` chưa dùng với `total + i ≥ target` hoặc `win[mask | bit] = False`.

Khác Cách 3: ta chỉ xét các mask **thực sự có thể đến**, không quét `2^n` mask.

In [17]:
def can_win_layers(n, target):
    if n * (n + 1) // 2 < target:
        return False
    total = {0: 0}
    layers = [[0]]
    while True:
        nxt = []
        for mask in layers[-1]:
            s = total[mask]
            for i in range(1, n + 1):
                bit = 1 << (i - 1)
                if mask & bit or s + i >= target:
                    continue                     # đã dùng, hoặc nước này thắng ngay (không sinh thế mới)
                child = mask | bit
                if child not in total:
                    total[child] = s + i
                    nxt.append(child)
        if not nxt:
            break
        layers.append(nxt)

    win = {}
    for layer in reversed(layers):
        for mask in layer:
            s = total[mask]
            res = False
            for i in range(1, n + 1):
                bit = 1 << (i - 1)
                if mask & bit:
                    continue
                if s + i >= target or not win[mask | bit]:
                    res = True
                    break
            win[mask] = res
    return win[0]


for (n, t), expected in examples:
    print((n, t), can_win_layers(n, t), "kỳ vọng", expected)

(10, 11) False kỳ vọng False
(10, 0) True kỳ vọng True
(10, 1) True kỳ vọng True


In [18]:
# Code nộp LeetCode — Cách 5
class Solution5:
    def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool:
        n, target = maxChoosableInteger, desiredTotal
        if n * (n + 1) // 2 < target:
            return False
        total = {0: 0}
        layers = [[0]]
        while True:
            nxt = []
            for mask in layers[-1]:
                s = total[mask]
                for i in range(1, n + 1):
                    bit = 1 << (i - 1)
                    if mask & bit or s + i >= target:
                        continue
                    child = mask | bit
                    if child not in total:
                        total[child] = s + i
                        nxt.append(child)
            if not nxt:
                break
            layers.append(nxt)
        win = {}
        for layer in reversed(layers):
            for mask in layer:
                s = total[mask]
                res = False
                for i in range(1, n + 1):
                    bit = 1 << (i - 1)
                    if not mask & bit and (s + i >= target or not win[mask | bit]):
                        res = True
                        break
                win[mask] = res
        return win[0]


print(Solution5().canIWin(10, 11), Solution5().canIWin(4, 6), Solution5().canIWin(3, 4))

False True False


**Độ phức tạp Cách 5:** `Time = O(R · n)` với `R ≤ 2^n` là số thế có thể đến; `Space = O(R)`. Không đệ quy nên **không lo giới hạn độ sâu** (dù `n ≤ 20` nên độ sâu đệ quy chỉ tối đa 20, vốn rất an toàn).

**Tóm tắt:** Cách 1 để hiểu luật; Cách 2 là bản nộp phổ biến; Cách 3 là DP "chuẩn sách"; Cách 4 thêm cắt tỉa; Cách 5 tách rõ hai giai đoạn xuôi/ngược.

## 8. TRACE từng bước

`trace_search` in cây tìm kiếm của Cách 2 với `(3, 4)`: mỗi dòng là một lời gọi `win(mask, total)`; thụt lề thể hiện độ sâu; đánh dấu khi dùng lại kết quả đã ghi nhớ. `trace_dp` in bảng DP từ dưới lên của Cách 3.

In [19]:
def trace_search(n, target):
    memo = {}
    depth = [0]

    def label(mask):
        return "{" + ",".join(str(i) for i in range(1, n + 1) if (mask >> (i - 1)) & 1) + "}"

    def win(mask, total):
        pad = "  " * depth[0]
        if mask in memo:
            print(f"{pad}win({label(mask)}, tổng={total}) = {memo[mask]}   (lấy từ memo)")
            return memo[mask]
        print(f"{pad}win({label(mask)}, tổng={total}) ?")
        result = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if total + i >= target:
                print(f"{pad}  chọn {i}: {total}+{i}={total + i} >= {target}  -> THẮNG NGAY")
                result = True
                break
            print(f"{pad}  chọn {i}: tổng -> {total + i}, xem đối thủ:")
            depth[0] += 1
            opp = win(mask | bit, total + i)
            depth[0] -= 1
            if not opp:
                print(f"{pad}  đối thủ THUA sau khi chọn {i} -> thế này THẮNG")
                result = True
                break
        memo[mask] = result
        print(f"{pad}=> win({label(mask)}) = {result}")
        return result

    return win(0, 0)


print(trace_search(3, 4))

win({}, tổng=0) ?
  chọn 1: tổng -> 1, xem đối thủ:
  win({1}, tổng=1) ?
    chọn 2: tổng -> 3, xem đối thủ:
    win({1,2}, tổng=3) ?
      chọn 3: 3+3=6 >= 4  -> THẮNG NGAY
    => win({1,2}) = True
    chọn 3: 1+3=4 >= 4  -> THẮNG NGAY
  => win({1}) = True
  chọn 2: tổng -> 2, xem đối thủ:
  win({2}, tổng=2) ?
    chọn 1: tổng -> 3, xem đối thủ:
    win({1,2}, tổng=3) = True   (lấy từ memo)
    chọn 3: 2+3=5 >= 4  -> THẮNG NGAY
  => win({2}) = True
  chọn 3: tổng -> 3, xem đối thủ:
  win({3}, tổng=3) ?
    chọn 1: 3+1=4 >= 4  -> THẮNG NGAY
  => win({3}) = True
=> win({}) = False
False


Đọc kết quả (`(3, 4)`): từ thế rỗng, A thử `1` → thế `{1}` (tổng 1) cho B. B thử `2`: tổng `3` chưa đủ, chuyển thế `{1,2}` cho A, và A chọn `3` được `6 ≥ 4` thắng nên `{1,2}` là **thế thắng của người đến lượt** (tức B không nên đi nước `2`); B thử tiếp `3` được `1 + 3 = 4 ≥ 4` **thắng ngay** ⇒ `win({1}) = True`: B thắng, nước `1` của A hỏng. Tương tự A thử `2` (B thắng bằng `3`: `2 + 3 = 5`) và `3` (B thắng bằng `1`: `3 + 1 = 4`). Mọi nước của A đều để B thắng ⇒ `win({}) = False`. Dòng "(lấy từ memo)" cho thấy thế `{1,2}` chỉ cần tính một lần dù được ghé hai lần (qua `1→2` và `2→1`): đó chính là điểm lợi của ghi nhớ.

In [20]:
def trace_dp(n, target):
    full = 1 << n
    total = [0] * full
    for mask in range(1, full):
        total[mask] = total[mask & (mask - 1)] + (mask & -mask).bit_length()
    win = [None] * full
    print(f"{'mask':>5} {'tập đã dùng':>12} {'tổng':>5}  kết quả")
    for mask in range(full - 1, -1, -1):
        used = [i for i in range(1, n + 1) if (mask >> (i - 1)) & 1]
        s = total[mask]
        if s >= target:
            print(f"{mask:0{n}b} {str(used):>12} {s:>5}  (đã kết thúc, bỏ qua)")
            continue
        win[mask] = False
        reason = "mọi nước đều để đối thủ thắng"
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if s + i >= target:
                win[mask] = True
                reason = f"chọn {i}: {s}+{i} >= {target}"
                break
            if not win[mask | bit]:
                win[mask] = True
                reason = f"chọn {i} -> đối thủ ở thế thua"
                break
        print(f"{mask:0{n}b} {str(used):>12} {s:>5}  win={win[mask]}  ({reason})")
    return win[0]


print(trace_dp(3, 4))

 mask  tập đã dùng  tổng  kết quả
111    [1, 2, 3]     6  (đã kết thúc, bỏ qua)
110       [2, 3]     5  (đã kết thúc, bỏ qua)
101       [1, 3]     4  (đã kết thúc, bỏ qua)
100          [3]     3  win=True  (chọn 1: 3+1 >= 4)
011       [1, 2]     3  win=True  (chọn 3: 3+3 >= 4)
010          [2]     2  win=True  (chọn 3: 2+3 >= 4)
001          [1]     1  win=True  (chọn 3: 1+3 >= 4)
000           []     0  win=False  (mọi nước đều để đối thủ thắng)
False


Bảng DP cho `(3, 4)`: các mask có tổng `≥ 4` là thế đã kết thúc (bỏ qua). Mask `{1,2}` (tổng 3): chọn `3` ⇒ `6 ≥ 4` ⇒ thắng. Mask `{1,3}`, `{2,3}`: tổng đã `≥ 4`, bỏ qua... riêng `{3}` (tổng 3): chọn `1` được `4` ⇒ thắng; `{1}` (tổng 1): chọn `3` được `4` ⇒ thắng; `{2}` (tổng 2): chọn `3` được `5` ⇒ thắng; mask rỗng: mọi nước dẫn đến thế thắng của đối thủ ⇒ **False**.

## 9. Code nộp LeetCode

Bản nộp chính: ghi nhớ theo mask (Cách 2). Ngắn, và nhờ cắt sớm khi gặp nước thắng nên rất nhanh.

In [21]:
class Solution:
    def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool:
        n, target = maxChoosableInteger, desiredTotal
        if n * (n + 1) // 2 < target:
            return False
        memo = {}

        def win(mask, total):
            if mask in memo:
                return memo[mask]
            result = False
            for i in range(1, n + 1):
                bit = 1 << (i - 1)
                if mask & bit:
                    continue
                if total + i >= target or not win(mask | bit, total + i):
                    result = True
                    break
            memo[mask] = result
            return result

        return win(0, 0)


for (n, t), expected in examples:
    assert Solution().canIWin(n, t) == expected
print("OK các ví dụ đề")

OK các ví dụ đề


## 10. Vì sao thuật toán đúng

**Mô hình.** Một thế là tập `U ⊆ {1..n}` đã dùng, tổng `s(U) = Σ U`. Người đến lượt chọn `i ∉ U`. Nếu `s(U) + i ≥ target` người đó thắng ngay; ngược lại chuyển sang thế `U ∪ {i}` cho đối thủ.

**Định nghĩa đệ quy đúng.** Gọi `W(U) = True` nếu người đến lượt ở thế `U` ép thắng được. Theo quy tắc minimax:

`W(U) = ∃ i ∉ U : ( s(U) + i ≥ target  hoặc  ¬W(U ∪ {i}) )`.

Nếu không còn `i` nào hoặc không nước nào thoả thì `W(U) = False`. Công thức này phản ánh chính xác "thắng nếu có nước thắng ngay hoặc đưa đối thủ vào thế thua". Vì mỗi nước đi thêm đúng một phần tử vào `U`, đệ quy **kết thúc** sau tối đa `n` bước.

**Vì sao chỉ cần `mask`.** `W(U)` chỉ phụ thuộc `U` (và hằng `target`): `s(U)` là hàm của `U`. Vậy hai đường dẫn khác thứ tự tới cùng `U` cho cùng `W`; ghi nhớ theo `U` (mask) là hợp lệ.

**Loại trường hợp "không ai đạt được".** Nếu `s({1..n}) < target`, trò chơi hết số mà không ai thắng; đề quy ước `false`. Nếu **không** chặn trường hợp này, công thức trên sẽ coi "thế hết số, không nước đi" là thua cho người đến lượt, khiến người vừa đi bị tính là **thắng** (sai). Vì vậy bước kiểm tra đầu hàm là bắt buộc. Khi `s({1..n}) ≥ target` thì mọi ván đều kết thúc có người thắng trước khi hết số (vì tổng cuối cùng `≥ target`), nên công thức đúng.

**Cách 3, 5** tính cùng `W` theo thứ tự đảm bảo con tính trước cha (mask lớn trước; tầng sau trước). **Cách 4:** nếu `max_chưa_dùng ≥ need` thì có nước thắng ngay nên `W = True`; ngược lại chạy đúng công thức, chỉ đổi biến `total` thành `need = target - total` và thứ tự thử.

Mục 12 đối chiếu mọi cách với một **oracle độc lập**: gán nhãn thế thắng/thua bằng **lặp đến điểm bất động** trên tập mọi tập con (không đệ quy, không memo).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Minimax vét cạn | `O(n!)` | `O(n)` | chỉ dùng cho `n ≤ 8` |
| 2. Ghi nhớ theo mask | `O(2^n · n)` | `O(2^n)` | **nộp bài**, thăm ít thế hơn tối đa |
| 3. DP từ dưới lên | `Θ(2^n · n)` | `Θ(2^n)` | quét hết mọi mask |
| 4. Ghi nhớ + cắt tỉa | `O(2^n · n)` | `O(2^n)` | số thế thăm tuỳ ca, có thể ít hoặc nhiều hơn Cách 2 |
| 5. Duyệt tầng + đánh giá ngược | `O(R · n)` | `O(R)` | `R` = số thế có thể đến |

Với `n = 20`, `2^n ≈ 10^6` thế, mỗi thế `≤ 20` nước đi: `~2·10^7` phép trong trường hợp xấu nhất.

## 12. Bộ test

### 12.1. Oracle độc lập + ví dụ đề + edge case

**Oracle** không đệ quy, không ghi nhớ: liệt kê **mọi tập con** `U` có `s(U) < target` bằng `itertools.combinations`, rồi **lặp** gán nhãn thắng/thua đến khi không còn thay đổi: một thế là *thắng* nếu có nước thắng ngay hoặc nước dẫn tới thế *thua*; *thua* nếu mọi nước đều dẫn tới thế *thắng* (và không có nước thắng ngay).

In [22]:
def oracle(n, target):
    if target <= 0:
        return True                 # chọn số nào cũng thắng ngay
    if n * (n + 1) // 2 < target:
        return False
    nums = range(1, n + 1)
    states = [frozenset(c) for k in range(n + 1) for c in combinations(nums, k) if sum(c) < target]
    label = {}                      # frozenset -> True (thắng) / False (thua)
    changed = True
    while changed:
        changed = False
        for st in states:
            if st in label:
                continue
            s = sum(st)
            moves = [i for i in nums if i not in st]
            if any(s + i >= target for i in moves):
                label[st] = True
                changed = True
                continue
            outcomes = [label.get(st | {i}) for i in moves]
            if any(o is False for o in outcomes):
                label[st] = True
                changed = True
            elif outcomes and all(o is True for o in outcomes):
                label[st] = False
                changed = True
    return label[frozenset()]


solutions = {
    "memo": can_win_memo,
    "dp": can_win_dp,
    "pruned": can_win_pruned,
    "layers": can_win_layers,
    "Solution": lambda n, t: Solution().canIWin(n, t),
}

edge = [
    ((10, 11), False),
    ((10, 0), True),
    ((10, 1), True),
    ((1, 1), True),          # chọn 1 là đủ
    ((1, 2), False),         # tổng tất cả = 1 < 2
    ((2, 3), False),         # tổng tất cả = 3 = target, n = 2 chẵn
    ((3, 4), False),
    ((3, 6), True),          # tổng tất cả = target, n lẻ => người đi đầu thắng
    ((4, 10), False),        # tổng tất cả = target, n chẵn => người đi sau thắng
    ((4, 6), True),
    ((3, 7), False),         # tổng tất cả 6 < 7
    ((5, 15), True),
    ((6, 21), False),
    ((20, 0), True),
    ((20, 300), False),      # tổng 210 < 300
]
for (n, t), expected in edge:
    exp = oracle(n, t)
    assert exp == expected, ("oracle", (n, t), exp, expected)
    if n <= 8:
        assert can_win_brute(n, t) == exp, ("brute", n, t)
    for name, fn in solutions.items():
        if name == "dp" and n > 16:
            continue
        assert fn(n, t) == exp, (name, (n, t), exp)
print("OK", len(edge), "ca: oracle, brute và mọi cách đều khớp đáp án tính tay")

OK 15 ca: oracle, brute và mọi cách đều khớp đáp án tính tay


Riêng `(2, 3)`: số `1, 2`, tổng tất cả `3 = target` (`n` chẵn). Người đi đầu chọn `1` (tổng 1) rồi đối thủ chọn `2` được `3` thắng; hoặc chọn `2` (tổng 2) rồi đối thủ chọn `1` được `3` thắng ⇒ **False** (bài test trên đã xác nhận bằng oracle).

Quy luật đẹp: khi `desiredTotal` **đúng bằng** `1 + 2 + … + n`, chỉ nước chọn số cuối cùng mới đạt đích nên người thắng là người đi **lượt thứ `n`**: người đi đầu thắng khi và chỉ khi `n` lẻ.

In [23]:
for n in range(1, 13):
    t = n * (n + 1) // 2
    assert oracle(n, t) == (n % 2 == 1) == can_win_memo(n, t), n
print("OK: khi desiredTotal = n(n+1)/2, người đi đầu thắng <=> n lẻ")

OK: khi desiredTotal = n(n+1)/2, người đi đầu thắng <=> n lẻ


### 12.2. Phản ví dụ cho các cách làm sai

Ba "lời giải nghe có lý" nhưng sai: (1) quên chặn `tổng tất cả < target`, (2) ghi nhớ theo **tổng** thay vì mask, (3) cho phép **chọn lại** số (luật trò chơi "100").

In [24]:
def wrong_no_sum_check(n, target):
    memo = {}
    def win(mask, total):
        if mask in memo:
            return memo[mask]
        result = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if total + i >= target or not win(mask | bit, total + i):
                result = True
                break
        memo[mask] = result
        return result
    return win(0, 0)                       # SAI: thiếu kiểm tra tổng tất cả < target

def wrong_memo_by_sum(n, target):
    if n * (n + 1) // 2 < target:
        return False
    memo = {}
    def win(mask, total):
        if total in memo:                  # SAI: khoá là tổng, không phải mask
            return memo[total]
        result = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if total + i >= target or not win(mask | bit, total + i):
                result = True
                break
        memo[total] = result
        return result
    return win(0, 0)

def wrong_reuse(n, target):
    memo = {}
    def win(need):
        if need in memo:
            return memo[need]
        result = any(i >= need or not win(need - i) for i in range(1, n + 1))
        memo[need] = result
        return result
    return win(target)                      # SAI: cho phép chọn lại số

print("quên chặn tổng :", wrong_no_sum_check(3, 7), "đúng:", oracle(3, 7))
bad_sum = next((n, t) for n in range(2, 9) for t in range(1, n * (n + 1) // 2 + 1) if wrong_memo_by_sum(n, t) != oracle(n, t))
print("ghi nhớ theo tổng:", bad_sum, wrong_memo_by_sum(*bad_sum), "đúng:", oracle(*bad_sum))
bad_reuse = next((n, t) for n in range(2, 9) for t in range(1, n * (n + 1) // 2 + 1) if wrong_reuse(n, t) != oracle(n, t))
print("cho chọn lại      :", bad_reuse, wrong_reuse(*bad_reuse), "đúng:", oracle(*bad_reuse))
assert wrong_no_sum_check(3, 7) != oracle(3, 7)

quên chặn tổng : True đúng: False
ghi nhớ theo tổng: (4, 9) True đúng: False
cho chọn lại      : (4, 9) True đúng: False


### 12.3. Đối chiếu vét cạn mọi cặp nhỏ

Mọi `n ≤ 8` và mọi `desiredTotal` từ `0` đến `n(n+1)/2 + 3`; so cả 5 cách (và brute force) với oracle.

In [25]:
count = 0
for n in range(1, 9):
    for t in range(0, n * (n + 1) // 2 + 4):
        exp = oracle(n, t)
        assert can_win_brute(n, t) == exp, ("brute", n, t)
        for name, fn in solutions.items():
            assert fn(n, t) == exp, (name, n, t, exp)
        count += 1
print("đã đối chiếu", count, "cặp (n, desiredTotal): khớp hoàn toàn")

đã đối chiếu 152 cặp (n, desiredTotal): khớp hoàn toàn


### 12.4. Ngẫu nhiên có seed (đúng ràng buộc) và ca lớn

Đối chiếu các cách với nhau cho `n` đến `20`, `desiredTotal` đến `300` (Cách 3 và Cách 5 chỉ chạy tới `n ≤ 14` cho nhanh). Đo thời gian các ca nặng (Cách 5 chỉ chạy khi `n ≤ 15` vì phải dựng mọi thế có thể đến).

In [26]:
random.seed(464)
for _ in range(300):
    n = random.randint(1, 20)
    t = random.randint(0, min(300, n * (n + 1) // 2 + 20))
    res = {can_win_memo(n, t), can_win_pruned(n, t), Solution().canIWin(n, t)}
    if n <= 14:
        res.add(can_win_dp(n, t))
        res.add(can_win_layers(n, t))
    assert len(res) == 1, (n, t, res)
    if n <= 9:
        assert res == {oracle(n, t)}
print("OK 300 cặp ngẫu nhiên")

for n, t in [(20, 0), (20, 210), (20, 209), (20, 150), (19, 190), (18, 171), (20, 100), (20, 300)]:
    out = []
    for name in ("memo", "pruned") + (("layers",) if n <= 15 else ()):
        t0 = time.perf_counter()
        r = solutions[name](n, t)
        out.append((name, r, round(time.perf_counter() - t0, 3)))
    assert len({o[1] for o in out}) == 1
    print((n, t), out)

OK 300 cặp ngẫu nhiên
(20, 0) [('memo', True, 0.0), ('pruned', True, 0.0)]
(20, 210) [('memo', False, 0.042), ('pruned', False, 0.05)]
(20, 209) [('memo', False, 0.037), ('pruned', False, 0.047)]


(20, 150) [('memo', False, 1.998), ('pruned', False, 1.6)]
(19, 190) [('memo', True, 0.014), ('pruned', True, 0.019)]
(18, 171) [('memo', False, 0.017), ('pruned', False, 0.018)]


(20, 100) [('memo', True, 0.418), ('pruned', True, 0.133)]
(20, 300) [('memo', False, 0.0), ('pruned', False, 0.0)]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên loại trường hợp `tổng tất cả < desiredTotal`

Khi không ai đạt được `desiredTotal`, đề quy định kết quả `false`. Nếu quên, đệ quy coi "hết số mà chưa thắng" là thua cho người đến lượt, nên người đi trước bị tính là **thắng**.

In [27]:
print("sai :", wrong_no_sum_check(3, 7))      # True
print("đúng:", Solution().canIWin(3, 7))      # False (1+2+3 = 6 < 7)

sai : True
đúng: False


**Đúng:** thêm `if n * (n + 1) // 2 < target: return False` ở đầu.

### Lỗi 2: Ghi nhớ theo **tổng** thay vì theo mask

Cùng tổng có thể ứng với **tập đã dùng khác nhau** (`1+4 = 2+3`), và hai thế đó cho kết quả khác nhau vì số còn lại khác nhau.

In [28]:
n, t = bad_sum
print((n, t), "sai:", wrong_memo_by_sum(n, t), "| đúng:", oracle(n, t))
print("ví dụ cùng tổng khác tập: {1,4} và {2,3} đều tổng", 1 + 4, "=", 2 + 3)

(4, 9) sai: True | đúng: False
ví dụ cùng tổng khác tập: {1,4} và {2,3} đều tổng 5 = 5


**Đúng:** khoá ghi nhớ là `mask` (hoặc tập đã dùng). Tổng là **hệ quả** của mask nên không cần (và không đủ) làm khoá.

### Lỗi 3: Nhầm với trò chơi "100" cho phép chọn lại

Nếu cho chọn lại, trạng thái chỉ là "còn thiếu bao nhiêu" và kết quả khác hẳn.

In [29]:
n, t = bad_reuse
print((n, t), "cho chọn lại:", wrong_reuse(n, t), "| không chọn lại (đề):", oracle(n, t))

(4, 9) cho chọn lại: True | không chọn lại (đề): False


**Đúng:** đọc kỹ luật "không dùng lại số đã chọn" ⇒ cần mask.

### Lỗi 4: Bộ nhớ đệm **toàn cục** dùng chung giữa nhiều test

`mask` có nghĩa khác nhau khi `n` hoặc `target` khác. Nếu `memo` (hoặc `@cache` ở module) sống qua nhiều lời gọi, kết quả của test trước làm hỏng test sau.

In [30]:
GLOBAL_MEMO = {}

def can_win_global_memo(n, target):
    if n * (n + 1) // 2 < target:
        return False
    def win(mask, total):
        if mask in GLOBAL_MEMO:            # SAI: dùng chung giữa các (n, target)
            return GLOBAL_MEMO[mask]
        result = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if total + i >= target or not win(mask | bit, total + i):
                result = True
                break
        GLOBAL_MEMO[mask] = result
        return result
    return win(0, 0)

print(can_win_global_memo(10, 11), "(đúng False)")
print(can_win_global_memo(4, 6), "(đúng", oracle(4, 6), ")")      # mask 0 đã bị nhớ từ test trước!
print(can_win_global_memo(3, 4), "(đúng", oracle(3, 4), ")")

False (đúng False)
False (đúng True )
False (đúng False )


**Đúng:** tạo `memo` **trong** hàm `canIWin` (như bản nộp), hoặc xoá bộ nhớ (`cache.cache_clear()`) ở đầu mỗi lời gọi.

### Lỗi 5: Lệch chỉ số bit (dùng bit `i` thay vì `i - 1`)

Nếu dùng `1 << i` cho số `i` thì số `n` cần bit `n`, nên mảng kích thước `1 << n` bị **thiếu một bit** ⇒ `IndexError` ở DP từ dưới lên.

In [31]:
def dp_off_by_one(n, target):
    full = 1 << n
    win = [False] * full
    for mask in range(full - 1, -1, -1):
        for i in range(1, n + 1):
            child = mask | (1 << i)        # SAI: nên là 1 << (i - 1)
            _ = win[child]                 # IndexError khi i = n
    return win[0]

try:
    dp_off_by_one(4, 6)
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Đúng:** số `i` ↔ bit `i - 1` (hoặc cấp mảng kích thước `1 << (n + 1)` nếu cố ý dùng bit `i`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"hai người luân phiên, chơi tối ưu, ai thắng"** ⇒ định nghĩa **thế thắng**: `tồn tại nước đi sao cho đối thủ ở thế thua`.
- Thấy **tập số nhỏ (`n ≤ 20`) và "không chọn lại"** ⇒ **mặt nạ bit + ghi nhớ**; số thế là `2^n`.
- Hỏi "**tổng hiện tại cần lưu không?**": nếu nó hoàn toàn quyết định bởi tập đã chọn thì **không** cần làm khoá.
- Nhớ hai ca đặc biệt: `desiredTotal ≤ maxChoosableInteger` ⇒ **True** (chọn số lớn nhất là xong); `tổng tất cả < desiredTotal` ⇒ **False**.
- Khi `desiredTotal = n(n+1)/2`: **đi đầu thắng ⇔ `n` lẻ** (mục 12.1).
- Cùng họ: 486 (Predict the Winner), 877 (Stone Game), 292 (Nim Game), 375 (Guess Number II), 698 (Partition to K Equal Sum Subsets, cũng bitmask).

## 15. Biến thể của bài

1. **Cho phép chọn lại** (trò chơi "100" gốc): đáp án có công thức: `true` nếu `desiredTotal <= maxChoosableInteger` hoặc `desiredTotal % (maxChoosableInteger + 1) != 0`.
2. **Liệt kê các nước đi đầu tiên thắng** (hoặc trả một nước thắng).
3. **Số thế (trạng thái) mà thuật toán phải thăm** — để đo độ nặng của đề.
4. **Misère:** người làm tổng đạt/vượt mục tiêu **thua** (đảo luật).

Dưới đây là biến thể 1, 2 và 4 (có test).

In [32]:
def can_win_reuse_formula(n, target):
    """Trò chơi '100' có chọn lại: kết quả bằng công thức (đã đối chiếu với vét cạn)."""
    return target <= n or target % (n + 1) != 0

def can_win_reuse_brute(n, target):
    memo = {}
    def win(need):
        if need in memo:
            return memo[need]
        r = any(i >= need or not win(need - i) for i in range(1, n + 1))
        memo[need] = r
        return r
    return win(target) if target > 0 else True

def winning_first_moves(n, target):
    """Các số i mà người đi đầu có thể chọn để ép thắng (list rỗng nếu thua)."""
    if n * (n + 1) // 2 < target:
        return []
    memo = {}
    def win(mask, total):
        if mask in memo:
            return memo[mask]
        r = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if total + i >= target or not win(mask | bit, total + i):
                r = True
                break
        memo[mask] = r
        return r
    res = []
    for i in range(1, n + 1):
        if i >= target or not win(1 << (i - 1), i):
            res.append(i)
    return res

def can_win_misere(n, target):
    """Misere: ai làm tổng >= target thì THUA. Nếu bị buộc phải làm vậy (hết số an toàn) thì thua."""
    memo = {}
    def win(mask, total):
        if mask in memo:
            return memo[mask]
        r = False
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit or total + i >= target:
                continue                    # nước làm tổng >= target là nước THUA, không chọn trừ khi bắt buộc
            if not win(mask | bit, total + i):
                r = True
                break
        memo[mask] = r
        return r
    return win(0, 0)

for n in range(1, 9):
    for t in range(1, 40):
        assert can_win_reuse_formula(n, t) == can_win_reuse_brute(n, t), (n, t)
        fm = winning_first_moves(n, t)
        assert (len(fm) > 0) == oracle(n, t), (n, t)
print(winning_first_moves(4, 6), winning_first_moves(10, 11), winning_first_moves(5, 9))
print(can_win_misere(3, 4), can_win_misere(4, 7), can_win_misere(2, 2))
print("OK biến thể")

[1] [] [3]
True True True
OK biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax vét cạn | thử mọi thứ tự chọn | `O(n!)` | `O(n)` | chỉ để hiểu luật, `n ≤ 8` |
| 2. Ghi nhớ theo mask | khoá = tập đã dùng | `O(2^n · n)` | `O(2^n)` | **nộp bài**, dễ viết |
| 3. DP từ dưới lên | mask giảm dần | `Θ(2^n · n)` | `Θ(2^n)` | thích vòng lặp, kiểm chứng |
| 4. Ghi nhớ + cắt tỉa | số lớn nhất ≥ còn thiếu ⇒ thắng | `O(2^n · n)` | `O(2^n)` | thử nghiệm heuristic; đo số thế thăm |
| 5. Tầng xuôi + đánh giá ngược | chỉ các thế có thể đến | `O(R · n)` | `O(R)` | tránh đệ quy, đếm số thế |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().canIWin(4, 6)`, `Solution().canIWin(3, 7)`, `Solution().canIWin(10, 0)`, `Solution().canIWin(4, 10)`. Điền `ans1` là list bốn giá trị `True`/`False` (hiện `None`).

**Bài 2 (tính tay).** Với `(4, 6)`: liệt kê từng nước đi đầu tiên `1, 2, 3, 4` và cho biết nước nào giúp người đi đầu **ép thắng**. Điền `ans2` là list các số thắng (hiện `None`).

**Bài 3 (điền TODO).** Hoàn thành `can_win_todo(n, target)` theo Cách 2.

In [33]:
ans1 = None
ans2 = None

def can_win_todo(n, target):
    # TODO 1: nếu n*(n+1)//2 < target: return False
    # TODO 2: memo = {}
    # TODO 3: def win(mask, total): nếu mask trong memo thì trả memo[mask]
    # TODO 4:     với i từ 1..n chưa dùng (bit i-1 của mask bằng 0):
    # TODO 5:         nếu total + i >= target hoặc not win(mask | (1 << (i-1)), total + i): result = True; break
    # TODO 6:     lưu memo[mask] = result và trả về
    # TODO 7: return win(0, 0)
    pass

# KHÔNG gọi can_win_todo ở đây để Run All không lỗi.

**Bài 4 (tự viết lại).** Viết `can_win_bottom_up(n, target)` theo Cách 3: mảng `total`, mảng `win`, duyệt mask giảm dần; nhớ bỏ qua mask có `total >= target` và trả `True` sớm khi `target <= n`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `can_win_reuse(n, target)` cho trò chơi **cho phép chọn lại** bằng **công thức** (không đệ quy). (b) Viết `count_winning_first_moves(n, target)` trả về **số** nước đi đầu tiên ép thắng. Gợi ý: (a) thế thua là `need` chia hết cho `n + 1`; (b) dùng ghi nhớ theo mask rồi thử từng nước đầu.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `(4, 6)` có tổng tất cả `10 ≥ 6`; `(3, 7)` có tổng `6 < 7`; `(10, 0)`: mục tiêu `0` thì chọn gì cũng thắng; `(4, 10)`: tổng tất cả đúng `10`, `n = 4` chẵn.
- **Mức 2 (vừa):** Bài 2: thử chọn `1`: còn thiếu `5`, đối thủ chọn gì thì người đi đầu cũng còn số lớn đủ để về đích? Hãy kiểm tra từng lựa chọn của đối thủ (`2, 3, 4`). Bài 3: bám sát mục 9.
- **Mức 3 (sâu):** Bài 4: `total[mask] = total[mask & (mask - 1)] + (mask & -mask).bit_length()`. Bài 5(a): `target <= n` thắng ngay; ngược lại thua khi `target % (n + 1) == 0` (người sau luôn bù cho đủ `n + 1`). 5(b): lặp `i` từ `1..n`; nước `i` tốt nếu `i >= target` hoặc `not win(mask của {i}, i)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
ans1 = [True, False, True, False]
assert ans1 == [Solution().canIWin(4, 6), Solution().canIWin(3, 7), Solution().canIWin(10, 0), Solution().canIWin(4, 10)]

# Bài 2: (4, 6): chỉ chọn 1 là ép thắng (xem trace: mọi lựa chọn khác để đối thủ thắng ngay)
ans2 = [1]
assert ans2 == winning_first_moves(4, 6)
print("OK bài 1-2; nước thắng của (4, 6) =", ans2)

OK bài 1-2; nước thắng của (4, 6) = [1]


Kiểm tra tay Bài 2 `(4, 6)`: chọn `1` (tổng 1, cần thêm `5`): đối thủ chọn `4` → 5 (chưa đủ), mình chọn `2` hoặc `3` → ≥ 6 thắng; đối thủ chọn `3` → 4, mình chọn `4` → 8 thắng; đối thủ chọn `2` → 3, mình chọn `4` → 7 thắng. Mọi nước của đối thủ đều thua ⇒ chọn `1` thắng. Chọn `2`: đối thủ chọn `4` → 6 thắng; chọn `3`: đối thủ chọn `4` → 7 thắng; chọn `4`: đối thủ chọn `2` → 6 thắng.

In [35]:
# Bài 3
def can_win_todo(n, target):
    if n * (n + 1) // 2 < target:
        return False
    memo = {}
    def win(mask, total):
        if mask in memo:
            return memo[mask]
        result = False
        for i in range(1, n + 1):
            if (mask >> (i - 1)) & 1:
                continue
            if total + i >= target or not win(mask | (1 << (i - 1)), total + i):
                result = True
                break
        memo[mask] = result
        return result
    return win(0, 0)

# Bài 4
def can_win_bottom_up(n, target):
    if n * (n + 1) // 2 < target:
        return False
    if target <= n:
        return True            # chọn ngay n; cũng xử lý target = 0
    full = 1 << n
    total = [0] * full
    for mask in range(1, full):
        total[mask] = total[mask & (mask - 1)] + (mask & -mask).bit_length()
    win = [False] * full
    for mask in range(full - 1, -1, -1):
        s = total[mask]
        if s >= target:
            continue
        for i in range(1, n + 1):
            bit = 1 << (i - 1)
            if mask & bit:
                continue
            if s + i >= target or not win[mask | bit]:
                win[mask] = True
                break
    return win[0]

for n in range(1, 9):
    for t in range(0, n * (n + 1) // 2 + 3):
        exp = oracle(n, t)
        assert can_win_todo(n, t) == can_win_bottom_up(n, t) == exp == Solution().canIWin(n, t), (n, t)
print("OK bài 3-4")

OK bài 3-4


In [36]:
# Bài 5(a): cho phép chọn lại: công thức
def can_win_reuse(n, target):
    return target <= n or target % (n + 1) != 0

# Bài 5(b): đếm các nước đi đầu ép thắng
def count_winning_first_moves(n, target):
    return len(winning_first_moves(n, target))

for n in range(1, 9):
    for t in range(1, 40):
        assert can_win_reuse(n, t) == can_win_reuse_brute(n, t), (n, t)
assert count_winning_first_moves(4, 6) == 1
assert count_winning_first_moves(10, 11) == 0
assert count_winning_first_moves(10, 1) == 10          # mục tiêu 1: chọn số nào cũng thắng ngay
for n in range(1, 8):
    for t in range(0, n * (n + 1) // 2 + 2):
        assert (count_winning_first_moves(n, t) > 0) == oracle(n, t)
print("OK bài 5")

OK bài 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (trò chơi hai người trên tập đã dùng: bitmask + ghi nhớ)

```python
class Solution:
    def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool:
        n, target = maxChoosableInteger, desiredTotal
        if n * (n + 1) // 2 < target:
            return False                          # không ai đạt được
        memo = {}                                 # khoá: mask (tập đã dùng)

        def win(mask, total):                     # người đến lượt có ép thắng được?
            if mask in memo:
                return memo[mask]
            result = False
            for i in range(1, n + 1):
                bit = 1 << (i - 1)
                if mask & bit:
                    continue
                if total + i >= target or not win(mask | bit, total + i):
                    result = True
                    break
            memo[mask] = result
            return result

        return win(0, 0)
```

**Checklist**

- [ ] Đã chặn `n(n+1)/2 < desiredTotal` ⇒ `False`?
- [ ] Khoá ghi nhớ là **mask** (không phải tổng), và `memo` tạo **trong** hàm (không dùng chung giữa test)?
- [ ] "Thắng" = có nước thắng ngay **hoặc** có nước dẫn đối thủ vào thế thua (`any`/`break`)?
- [ ] Số `i` ứng với bit `i - 1` (không lệch chỉ số)?
- [ ] Đã thử `(10,11)` False, `(10,0)` True, `(10,1)` True, `(3,4)` False, `(3,7)` False, `(4,6)` True?
- [ ] Skeleton đúng `class Solution` / `def canIWin(self, maxChoosableInteger: int, desiredTotal: int) -> bool`?